In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")

train_data.head()
# train_data.isnull().sum()
train_data["Cabin"] = train_data['Cabin'].fillna(0)
train_data.loc[train_data['Cabin'] != 0, 'Cabin'] = 1
train_data["FamilySize"] = train_data["SibSp"] + train_data["Parch"]
train_data.loc[train_data['FamilySize'] != 0, 'isAlone'] = 1
train_data['isAlone'] = train_data['isAlone'].fillna(0)

# filling in empty ages
train_data['Title'] = train_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

train_data["Age"] = train_data["Age"].fillna(train_data.groupby("Title")["Age"].transform("mean"))

In [3]:
#need to predict Survived column
# test_data.head()
# survived = train_data[
#     (train_data["Sex"] == "male") &
#     (train_data["Survived"] == 1)
# ].shape[0]

#need to transform all text data to numbers
# train_data.loc[train_data["Sex"] == "female","Sex"] = 1
# train_data.loc[train_data["Sex"] == "male","Sex"] = 0
train_data["Sex"] = train_data["Sex"].map({"male": 0, "female": 1})

# train_data.loc[train_data["Embarked"] == "S", "Embarked"] = 0
# train_data.loc[train_data["Embarked"] == "Q", "Embarked"] = 1
# train_data.loc[train_data["Embarked"] == "C", "Embarked"] = 2
train_data["Embarked"] = train_data['Embarked'].fillna(train_data['Embarked'].mode()[0])

embarked_dummies = pd.get_dummies(train_data["Embarked"], prefix="Embarked", drop_first=True)

train_data = pd.concat([train_data, embarked_dummies], axis=1)
train_data = train_data.drop("Embarked", axis=1)
train_data = train_data.drop("Name", axis=1)
train_data = train_data.drop("PassengerId", axis=1)
# train_data = train_data.drop("Embarked", axis=1)

# total = train_data[
#     (train_data["Sex"] == "male") 
#     # (train_data["Survived"] == 0)
# ].shape[0]

# print(survived/total)


train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,FamilySize,isAlone,Title,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,A/5 21171,7.2500,0,1,1.0,Mr,False,True
1,1,1,1,38.0,1,0,PC 17599,71.2833,1,1,1.0,Mrs,False,False
2,1,3,1,26.0,0,0,STON/O2. 3101282,7.9250,0,0,0.0,Miss,False,True
3,1,1,1,35.0,1,0,113803,53.1000,1,1,1.0,Mrs,False,True
4,0,3,0,35.0,0,0,373450,8.0500,0,0,0.0,Mr,False,True


In [4]:
title_dummies = pd.get_dummies(train_data["Title"], prefix="Title", drop_first=True)
train_data = pd.concat([train_data, title_dummies], axis=1)
train_data = train_data.drop("Title", axis=1)
train_data = train_data.drop("Cabin", axis=1)
train_data = train_data.drop("Ticket", axis=1)
X_train = train_data.drop("Survived", axis=1)
Y_train = train_data["Survived"]

In [5]:
test_data_original = test_data
# handle test data
test_data["Cabin"] = test_data['Cabin'].fillna(0)
test_data.loc[test_data['Cabin'] != 0, 'Cabin'] = 1
test_data["FamilySize"] = test_data["SibSp"] + test_data["Parch"]
test_data.loc[test_data['FamilySize'] != 0, 'isAlone'] = 1
test_data['isAlone'] = test_data['isAlone'].fillna(0)
test_data["Fare"] = test_data["Fare"].fillna(test_data["Fare"].median())

# filling in empty ages
test_data['Title'] = test_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

test_data["Age"] = test_data["Age"].fillna(test_data.groupby("Title")["Age"].transform("mean"))
test_data["Age"] = test_data["Age"].fillna(test_data["Age"].median())

test_data["Sex"] = test_data["Sex"].map({"male": 0, "female": 1})

# train_data.loc[train_data["Embarked"] == "S", "Embarked"] = 0
# train_data.loc[train_data["Embarked"] == "Q", "Embarked"] = 1
# train_data.loc[train_data["Embarked"] == "C", "Embarked"] = 2
test_data["Embarked"] = test_data['Embarked'].fillna(test_data['Embarked'].mode()[0])

embarked_dummies = pd.get_dummies(test_data["Embarked"], prefix="Embarked", drop_first=True)

test_data = pd.concat([test_data, embarked_dummies], axis=1)
test_data = test_data.drop("Embarked", axis=1)
test_data = test_data.drop("Name", axis=1)
test_data = test_data.drop("PassengerId", axis=1)

title_dummies = pd.get_dummies(test_data["Title"], prefix="Title", drop_first=True)
test_data = pd.concat([test_data, title_dummies], axis=1)
test_data = test_data.drop("Title", axis=1)
test_data = test_data.drop("Cabin", axis=1)
test_data = test_data.drop("Ticket", axis=1)


In [6]:
# test_data.isnull().sum()
clf = LogisticRegression(max_iter=10000, random_state=0)
clf.fit(X_train, Y_train)

X_test = test_data
# # Y_test = test_data["Survived"]
# test_data.head()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
predictions = clf.predict(X_test)
submission = pd.DataFrame({
    "PassengerId": test_data_original["PassengerId"],
    "Survived": predictions
})
submission.to_csv("submission.csv", index=False)
# acc = accuracy_score(y_test, clf.predict(X_test)) * 100
# print(f"Logistic Regression model accuracy: {acc:.2f}%")
